In [1]:
import torch
from PIL import Image, ImageDraw, ImageFilter
from diffusers import Flux2KleinInpaintPipeline


MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"


def resize_reference(image: Image.Image, max_side: int = 768) -> Image.Image:
    """Small full-image reference for global style/context."""
    image = image.convert("RGB")
    w, h = image.size

    scale = min(1.0, max_side / max(w, h))
    w2 = max(16, round(w * scale / 16) * 16)
    h2 = max(16, round(h * scale / 16) * 16)

    return image.resize((w2, h2), Image.Resampling.LANCZOS)


def expand_box(box, image_size, factor=2.5):
    """
    Expand a box around its center.
    factor=2.5 means the context crop is ~2.5x the original box size.
    """
    x1, y1, x2, y2 = box
    W, H = image_size

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    bw = x2 - x1
    bh = y2 - y1

    new_w = bw * factor
    new_h = bh * factor

    nx1 = max(0, int(cx - new_w / 2))
    ny1 = max(0, int(cy - new_h / 2))
    nx2 = min(W, int(cx + new_w / 2))
    ny2 = min(H, int(cy + new_h / 2))

    return nx1, ny1, nx2, ny2


def expand_edit_box(box, image_size, spill=0.15):
    """
    Slightly expand the actual editable region.

    This is important: if the mask is exactly the user box, Klein cannot
    naturally generate pixels outside that box.

    spill=0.15 gives ~15% extra space on each side.
    """
    x1, y1, x2, y2 = box
    W, H = image_size

    bw = x2 - x1
    bh = y2 - y1

    dx = bw * spill
    dy = bh * spill

    return (
        max(0, int(x1 - dx)),
        max(0, int(y1 - dy)),
        min(W, int(x2 + dx)),
        min(H, int(y2 + dy)),
    )


def load_klein():
    if torch.cuda.is_available():
        dtype = torch.bfloat16
    elif torch.backends.mps.is_available():
        dtype = torch.float16
    else:
        dtype = torch.float32

    pipe = Flux2KleinInpaintPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
    )

    if torch.cuda.is_available():
        # Replace with pipe.to("cuda") if you have enough VRAM.
        pipe.enable_model_cpu_offload()

    elif torch.backends.mps.is_available():
        pipe.to("mps")

    else:
        pipe.to("cpu")

    return pipe


def generate_object(
    pipe,
    image: Image.Image,
    box,
    prompt: str,
    *,
    context_factor=2.5,
    spill=0.15,
    reference_size=768,
    feather=8,
    seed=42,
):
    """
    box = (x1, y1, x2, y2) in original-image pixel coordinates.
    """

    image = image.convert("RGB")

    # ---------------------------------------------------------
    # 1. Large local crop around user's box.
    # ---------------------------------------------------------
    crop_box = expand_box(
        box,
        image.size,
        factor=context_factor,
    )

    cx1, cy1, cx2, cy2 = crop_box
    local = image.crop(crop_box)

    # ---------------------------------------------------------
    # 2. Allow generation to extend slightly outside user box.
    #
    # User box = placement intent.
    # Expanded box = actual editable region.
    # ---------------------------------------------------------
    edit_box_global = expand_edit_box(
        box,
        image.size,
        spill=spill,
    )

    ex1, ey1, ex2, ey2 = edit_box_global

    # Convert global coordinates -> local crop coordinates.
    edit_box_local = (
        ex1 - cx1,
        ey1 - cy1,
        ex2 - cx1,
        ey2 - cy1,
    )

    mask = Image.new("L", local.size, 0)
    draw = ImageDraw.Draw(mask)
    draw.rectangle(edit_box_local, fill=255)

    # Slight blur avoids a hard inpaint boundary.
    if feather > 0:
        mask_for_model = mask.filter(
            ImageFilter.GaussianBlur(feather)
        )
    else:
        mask_for_model = mask

    # ---------------------------------------------------------
    # 3. Whole image provides global style/context.
    # ---------------------------------------------------------
    global_reference = resize_reference(
        image,
        max_side=reference_size,
    )

    # ---------------------------------------------------------
    # 4. Make local inference dimensions multiples of 16.
    # ---------------------------------------------------------
    lw, lh = local.size

    model_w = max(16, round(lw / 16) * 16)
    model_h = max(16, round(lh / 16) * 16)

    local_model = local.resize(
        (model_w, model_h),
        Image.Resampling.LANCZOS,
    )

    mask_model = mask_for_model.resize(
        (model_w, model_h),
        Image.Resampling.LANCZOS,
    )

    # ---------------------------------------------------------
    # 5. Klein inpainting.
    # ---------------------------------------------------------
    full_prompt = (
        f"Add {prompt} naturally into the masked region. "
        "Make it belong naturally to the existing scene. "
        "Match the source image's visual style, lighting direction, "
        "lighting intensity, color temperature, perspective, camera "
        "characteristics, depth of field, texture detail, sharpness, "
        "and image grain. Preserve the surrounding scene."
    )

    generator = torch.Generator(device="cpu").manual_seed(seed)

    result = pipe(
        prompt=full_prompt,
        image=local_model,
        mask_image=mask_model,
        image_reference=global_reference,
        strength=1.0,
        num_inference_steps=4,
        guidance_scale=1.0,
        generator=generator,
    ).images[0]

    # Back to original crop resolution.
    result = result.resize(
        local.size,
        Image.Resampling.LANCZOS,
    )

    # ---------------------------------------------------------
    # 6. Explicitly preserve everything outside editable region.
    # ---------------------------------------------------------
    composite_mask = mask.filter(
        ImageFilter.GaussianBlur(feather)
    )

    local_result = Image.composite(
        result,
        local,
        composite_mask,
    )

    # ---------------------------------------------------------
    # 7. Paste local result back into untouched original.
    # ---------------------------------------------------------
    output = image.copy()
    output.paste(local_result, (cx1, cy1))

    return output, local, mask, global_reference

In [ ]:
pipe = load_klein()

image = Image.open("input.png")

# Example user-selected box
box = (420, 300, 700, 560)

result, local_crop, mask, reference = generate_object(
    pipe,
    image,
    box,
    prompt="a red Porsche parked on the ground",
    context_factor=2.5,
    spill=0.15,
    seed=42,
)

result.save("result.png")
local_crop.save("debug_crop.png")
mask.save("debug_mask.png")
reference.save("debug_reference.png")